# Downtown Edmonton Business Classifier

This notebook classifies companies as **downtown Edmonton** or not based on their Canadian postal codes.

### How it works
Canadian postal codes follow the format `A1A 1A1`. The first three characters form the **Forward Sortation Area (FSA)**, which maps to a specific geographic zone. The FSAs for downtown Edmonton are:

| FSA | Neighbourhood |
|-----|---------------|
| T5H | Downtown core (south) |
| T5J | Downtown core (financial/central district) |
| T5K | Oliver neighbourhood (immediately west of downtown) |

In [ ]:
# Install dependencies if needed
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', 'pandas', '-q'])

In [ ]:
import pandas as pd

# ── Configuration ────────────────────────────────────────────────────────────
DATA_FILE = 'Office_zoom_contact_list.xlsx'

# FSAs (first 3 characters of postal code) that cover downtown Edmonton.
# Adjust this set if you want to widen or narrow the definition of "downtown".
DOWNTOWN_EDMONTON_FSAS = {'T5H', 'T5J', 'T5K'}
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_excel(DATA_FILE)
print(f'Loaded {len(df)} rows')
df.head()

In [ ]:
def extract_fsa(postal_code) -> str | None:
    """Return the 3-char FSA from a Canadian postal code, or None if invalid."""
    if pd.isna(postal_code):
        return None
    cleaned = str(postal_code).strip().upper().replace('-', ' ')
    # Accept 'A1A 1A1' or 'A1A1A1'
    fsa = cleaned[:3]
    # Basic validation: letter-digit-letter
    if len(fsa) == 3 and fsa[0].isalpha() and fsa[1].isdigit() and fsa[2].isalpha():
        return fsa
    return None


def is_downtown_edmonton(postal_code) -> bool | None:
    """Return True if the postal code falls within downtown Edmonton.
    Returns None when the postal code is missing or unparseable.
    """
    fsa = extract_fsa(postal_code)
    if fsa is None:
        return None
    return fsa in DOWNTOWN_EDMONTON_FSAS


# Derive columns
postal_col = df.columns[1]          # 'Company Zip Code'
company_col = df.columns[0]         # 'Company Name'

df['FSA'] = df[postal_col].apply(extract_fsa)
df['Is Downtown Edmonton'] = df[postal_col].apply(is_downtown_edmonton)

df[[company_col, postal_col, 'FSA', 'Is Downtown Edmonton']].head(10)

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
total        = len(df)
downtown     = df['Is Downtown Edmonton'].sum()
not_downtown = (df['Is Downtown Edmonton'] == False).sum()
no_data      = df['Is Downtown Edmonton'].isna().sum()

print(f'Total companies   : {total}')
print(f'Downtown Edmonton : {int(downtown)}')
print(f'Not downtown      : {int(not_downtown)}')
print(f'No postal code    : {int(no_data)}')

In [ ]:
# ── Downtown companies ────────────────────────────────────────────────────────
print('Companies in downtown Edmonton:')
downtown_df = df[df['Is Downtown Edmonton'] == True][[company_col, postal_col, 'FSA']]
downtown_df = downtown_df.drop_duplicates(subset=company_col).reset_index(drop=True)
downtown_df

In [ ]:
# ── Full results table ────────────────────────────────────────────────────────
result = df[[company_col, postal_col, 'FSA', 'Is Downtown Edmonton']].copy()
result['Is Downtown Edmonton'] = result['Is Downtown Edmonton'].map(
    {True: 'Yes', False: 'No', None: 'Unknown'}
).fillna('Unknown')
result

In [ ]:
# ── Export results ────────────────────────────────────────────────────────────
out_file = 'downtown_edmonton_results.xlsx'
result.to_excel(out_file, index=False)
print(f'Results saved to {out_file}')